# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

## Load modules

In [1]:
# Module Imports
import pandas as pd            # For data manipulation and analysis
from bs4 import BeautifulSoup  # For parsing HTML content
import time                    # For managing time delays during scraping
import tqdm                    # For displaying progress bars during scraping
import pprint                  # For neatly displaying JSON cod
import re                      # For pattern recognition in extracted HTML
import os                      # For interacting with the operating system (e.g., file paths, environment variables)
import json
from afinn import Afinn


The below cell can be used to load the saved data:

In [2]:
# Directory and file names
output_dir = 'Scraping_files'
normal_output_filename = os.path.join(output_dir, 'html_content_normal.txt')

# Check if the normal file exists before attempting to open it
if not os.path.exists(normal_output_filename):
    raise FileNotFoundError(f"The file {normal_output_filename} does not exist.")

# Load normal data directly
with open(normal_output_filename, 'r', encoding='utf-8') as normal_file:
    # Read the entire file content
    raw_content = normal_file.read()

# Split the content into documents based on the delimiter
# We add `</html>` to ensure each document ends with this tag
list_htmls = [doc.strip() + '\n\n</html>' for doc in raw_content.split('</html>\n\n') if doc.strip()]

# Print the number of documents and the first item in the list for verification
#print(f"Number of HTML documents: {len(list_htmls)}")
#print("First item in HTML documents:")
print(list_htmls[0] if list_htmls else "No data in raw file")

# For testing
#list_htmls = list_htmls[0:5]

<!DOCTYPE html>
<html lang="da-DK">
<head>
    <title>Afskaf inklusionsloven</title>


    <meta charset="utf-8" />

    
<script id="Cookiebot" data-cbid="51f634e9-7d87-4212-af57-edd0e26f6f06" data-blockingmode="none" type="text/javascript" src="https://consent.cookiebot.com/uc.js"></script>

<script type="text/javascript">
       window.__THIRD_PARTY_KEYS = { sentry: "https://984e0a1cc92a49acaf5b315f1f3f1cd1@sentry.io/216509" };

       window.addEventListener('CookiebotOnAccept', function (e) {
           if (Cookiebot.consent.statistics) {
               // load app insight after consent
               var appInsights = window.appInsights || function (a) {
                   function b(a) { c[a] = function () { var b = arguments; c.queue.push(function () { c[a].apply(c, b) }) } } var c = { config: a }, d = document, e = window; setTimeout(function () { var b = d.createElement("script"); b.src = a.url || "https://az416426.vo.msecnd.net/scripts/a/ai.0.js", d.getElementsByTagName("scr

We now have a list for all the borgerforslag. We now proceed to take out the data from the html:

In [3]:
import json
from bs4 import BeautifulSoup
import pandas as pd

# Assuming list_htmls contains your HTML strings
# List to store the extracted data
extracted_data = []

# Loop through each HTML content and extract the required information
for html_content in list_htmls:
    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, 'lxml')
    
    # Extract the title
    title = soup.title.string if soup.title else "Title not found."
    
    # Locate the relevant section that contains the start date, end date, and votes
    date_section = soup.find('div', class_='_3l86Vg')
    start_date = end_date = votes = "Not found"
    
    if date_section:
        date_info = date_section.find_all('div')
        for info in date_info:
            if 'Startdato' in info.text:
                start_date = info.find('strong').text
            if 'Slutdato' in info.text:
                end_date = info.find('strong').text
            if 'Antal støtter' in info.text:
                votes = info.find('strong').text
    
    # Initialize external ID
    external_id = "Not found"
    
    # Extract the main body, remarks, and external ID from the JSON data within the script tag
    script_tag = soup.find('script', {'data-module': 'ProposalEditor'})
    main_body = ""
    remarks = ""
    
    proposers = []
    coproposers = []
    num_coauthors = 0  # Initialize the number of coauthors

    if script_tag:
        # Extract the JavaScript content as a string
        script_content = script_tag.string
        
        # Extract the JSON part from the script content
        json_str = script_content.split('_components.push(')[-1].rstrip(');')
        
        # Parse the JSON data
        data = json.loads(json_str)
        
        # Extract the 'proposalContent' text
        main_body = data['props']['proposalCreationViewModel']['proposal']['proposalContent']
        
        # Extract the 'remarks' text
        remarks = data['props']['proposalCreationViewModel']['proposal']['remarks']
        
        # Extract the 'externalId'
        external_id = data['props']['proposalCreationViewModel']['proposal'].get('externalId', 'Not found')
        
        # Extract main proposer details (assuming the possibility of multiple proposers)
        main_proposers = data['props']['proposalCreationViewModel']['proposal'].get('proposers', [data['props']['proposalCreationViewModel']['proposal']['author']])
        for proposer in main_proposers:
            proposers.append({
                'name': proposer['name'],
                'munici': proposer.get('city', 'Not provided')  # Use get to safely access 'city'
            })
        
        # Extract co-proposer details (if any)
        coauthors = data['props']['proposalCreationViewModel']['proposal']['coAuthors']
        
        # Count the number of coauthors
        num_coauthors = len(coauthors)
        
        # Loop through each co-author and capture their details
        for coauthor in coauthors:
            coproposers.append({
                'name': coauthor['name'],
                'munici': coauthor.get('city', 'Not provided')  # Use get to safely access 'city'
            })

    # Flatten the proposer details into a single dictionary
    proposers_flat = {}
    for i, proposer in enumerate(proposers, start=1):
        proposers_flat[f'Proposer{i}_munici'] = proposer['munici']

    # Flatten the co-proposer details into a single dictionary
    coproposers_flat = {}
    for i, coproposer in enumerate(coproposers, start=1):
        coproposers_flat[f'Coproposer{i}_munici'] = coproposer['munici']

    # Combine all the data into one dictionary
    combined_data = {
        'Title': title,
        'Start Date': start_date,
        'End Date': end_date,
        'Votes': votes,
        'File Number': external_id,
        'Main Body': main_body,
        'Remarks': remarks,
        'Num Coauthors': num_coauthors,  # Add the count of coauthors
        **proposers_flat,
        **coproposers_flat
    }
    
    # Append the extracted data to the list
    extracted_data.append(combined_data)

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(extracted_data)

# Display the DataFrame
df


,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,Coproposer3_munici,Coproposer4_munici,Coproposer5_munici,Coproposer6_munici,Coproposer7_munici,Coproposer8_munici,Coproposer9_munici,Coproposer10_munici
0,Afskaf inklusionsloven,15. august 2024,11. februar 2025,39,FT-18153,For at forbedre undervisningen og sikre optima...,"Inklusionsloven, som blev indført i 2012 med d...",3,Aarhus,Aarhus,Esbjerg,Esbjerg,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fartbøder forhindrer i at opnå Dansk statsborg...,06. august 2024,02. februar 2025,82,FT-18099,Forslaget drejer sig om regler for fartbøder v...,Vi synes ikke det er rimeligt at en fartbøde s...,3,Vejle,Vejle,Vejle,Vejle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Forbyd dressurridning som konkurrencesport,06. august 2024,02. februar 2025,245,FT-18075,Dressurridning som konkurrencesport er en spor...,Dette borgerforslag er stillet at dyreetiske å...,3,Hvidovre,Adressebeskyttelse,Tårnby,Frederiksberg,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Lavere skat og fjernelse af minimumsalder på p...,06. august 2024,02. februar 2025,338,FT-18046,Vi stiller et forslag om ændring af reglerne f...,Vi stiller dette forslag for at lette den økon...,3,Randers,København,Randers,Randers,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bloddonorpligt. (Som værnepligt),26. juli 2024,22. januar 2025,41,FT-18044,Jeg forslår at der etableres en pligt til at a...,Det er jo velkendt at danske regioner mangler ...,3,Hjørring,Hjørring,Hjørring,Hjørring,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,30. januar 2018,29. juli 2018,189,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,"Af hensyn til forurening.\nAf hensyn til at ""4...",3,Odsherred,Odsherred,Odsherred,Adressebeskyttelse,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1843,Automatisk førtidspension til personer der har...,30. januar 2018,29. juli 2018,66,FT-00059,Forslaget er at man automatisk giver førtidspe...,Formålet med lovforslaget er at sikre ofrene f...,4,Vejen,Sønderborg,Sønderborg,Kolding,Sønderborg,NaN,NaN,NaN,NaN,NaN,NaN
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,30. januar 2018,29. juli 2018,597,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",dette er blot en ud af alt for mange anbragte ...,3,Kalundborg,Kalundborg,Kalundborg,Sorø,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1845,"Statsborgerskab til unge mennesker, som er fød...",30. januar 2018,29. juli 2018,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,"Mennesker som er født og opvokset i Danmark, o...",3,Stevns,København,Næstved,Roskilde,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We check what data has been actually scraped:

In [4]:
# Define the path to the test.txt file in the Scraping_files directory
test_file_path = os.path.join('Scraping_files', 'test.txt')

# Step 1: Read the IDs from test.txt
with open(test_file_path, 'r') as file:
    test_urls = file.readlines()

# Extract IDs from the URLs in test.txt
test_ids = [url.split('Id=')[-1].strip() for url in test_urls]

# Step 2: Get the unique IDs from the DataFrame 'df'
df_ids = df['File Number'].unique()

# Step 3: Find IDs that are in test.txt but not in the DataFrame 'df'
missing_ids_in_df = set(test_ids) - set(df_ids)

# Step 4: Display the missing IDs
print("Missing IDs in DataFrame:")
for missing_id in missing_ids_in_df:
    print(missing_id)


Missing IDs in DataFrame:


Now we append merge the log data to get the times for which the data was scraped:

In [5]:
# Define the directory and file name
directory = "Scraping_files"
file_name = "logfile_borgerforslag.csv"

# Specify the run_id to filter the log entries
run_id = 20240818203552  # Replace with the actual numerical run_id you want to filter by

# Construct the full file path
file_path = os.path.join(directory, file_name)

# Read the CSV log file into a DataFrame, considering the run_id in the log format
log_df = pd.read_csv(file_path, sep=';', header=0, names=['Run ID', 'Timestamp', 'Status Code', 'Size', 'URL', 'File Path'])

# Ensure the 'Run ID' column is treated as numerical (if not already)
log_df['Run ID'] = pd.to_numeric(log_df['Run ID'], errors='coerce')

# Filter out any rows where the Timestamp is not in the expected format
log_df = log_df[log_df['Timestamp'].str.match(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}')]

# Convert the Timestamp to datetime
log_df['Timestamp'] = pd.to_datetime(log_df['Timestamp'], format='%Y-%m-%d %H:%M:%S')

# Extract the ID from the URL
log_df['ID'] = log_df['URL'].str.extract(r'Id=([A-Za-z0-9\-]+)')

# Extract the date part from the timestamp
log_df['Date'] = log_df['Timestamp'].dt.date

# Filter the DataFrame by run_id, ensuring you select the relevant entries
if run_id:
    log_df = log_df[log_df['Run ID'] == run_id]

# Sort the DataFrame by ID and Timestamp
log_df = log_df.sort_values(by=['ID', 'Timestamp'], ascending=[True, False])

# Drop duplicates, keeping only the latest entry for each ID on the same day
log_df_latest = log_df.drop_duplicates(subset=['ID', 'Date'], keep='first')

# Display the filtered and processed DataFrame
log_df_latest


,Run ID,Timestamp,Status Code,Size,URL,File Path,ID,Date
5635,20240818203552,2024-08-18 21:31:04,200,39309,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-00005,2024-08-18
5633,20240818203552,2024-08-18 21:31:00,200,37363,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-00034,2024-08-18
5616,20240818203552,2024-08-18 21:30:27,200,35951,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-00042,2024-08-18
5632,20240818203552,2024-08-18 21:30:58,200,36987,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-00059,2024-08-18
5594,20240818203552,2024-08-18 21:29:49,200,36555,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-00062,2024-08-18
...,...,...,...,...,...,...,...,...
3798,20240818203552,2024-08-18 20:36:07,200,40852,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-18050,2024-08-18
3800,20240818203552,2024-08-18 20:36:11,200,38849,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-18062,2024-08-18
3792,20240818203552,2024-08-18 20:35:56,200,41205,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-18075,2024-08-18
3791,20240818203552,2024-08-18 20:35:54,200,40098,https://borgerforslag.dk/se-og-stoet-forslag/?...,c:\Users\olive\OneDrive - Københavns Universit...,FT-18099,2024-08-18


The merge:

In [6]:
# Perform the join on 'File Number' from df and 'ID' from log_df_latest
merged_df = pd.merge(df, log_df_latest[['ID', 'Timestamp']], left_on='File Number', right_on='ID', how='left')

# Drop the 'ID' column and rename 'Timestamp' to 'Scrape'
merged_df = merged_df.drop(columns=['ID']).rename(columns={'Timestamp': 'Scrape'})

# Display the merged DataFrame
merged_df

,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,Coproposer3_munici,Coproposer4_munici,Coproposer5_munici,Coproposer6_munici,Coproposer7_munici,Coproposer8_munici,Coproposer9_munici,Coproposer10_munici,Scrape
0,Afskaf inklusionsloven,15. august 2024,11. februar 2025,39,FT-18153,For at forbedre undervisningen og sikre optima...,"Inklusionsloven, som blev indført i 2012 med d...",3,Aarhus,Aarhus,Esbjerg,Esbjerg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 20:35:52
1,Fartbøder forhindrer i at opnå Dansk statsborg...,06. august 2024,02. februar 2025,82,FT-18099,Forslaget drejer sig om regler for fartbøder v...,Vi synes ikke det er rimeligt at en fartbøde s...,3,Vejle,Vejle,Vejle,Vejle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 20:35:54
2,Forbyd dressurridning som konkurrencesport,06. august 2024,02. februar 2025,245,FT-18075,Dressurridning som konkurrencesport er en spor...,Dette borgerforslag er stillet at dyreetiske å...,3,Hvidovre,Adressebeskyttelse,Tårnby,Frederiksberg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 20:35:56
3,Lavere skat og fjernelse af minimumsalder på p...,06. august 2024,02. februar 2025,338,FT-18046,Vi stiller et forslag om ændring af reglerne f...,Vi stiller dette forslag for at lette den økon...,3,Randers,København,Randers,Randers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 20:35:58
4,Bloddonorpligt. (Som værnepligt),26. juli 2024,22. januar 2025,41,FT-18044,Jeg forslår at der etableres en pligt til at a...,Det er jo velkendt at danske regioner mangler ...,3,Hjørring,Hjørring,Hjørring,Hjørring,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 20:36:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,30. januar 2018,29. juli 2018,189,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,"Af hensyn til forurening.\nAf hensyn til at ""4...",3,Odsherred,Odsherred,Odsherred,Adressebeskyttelse,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 21:30:56
1843,Automatisk førtidspension til personer der har...,30. januar 2018,29. juli 2018,66,FT-00059,Forslaget er at man automatisk giver førtidspe...,Formålet med lovforslaget er at sikre ofrene f...,4,Vejen,Sønderborg,Sønderborg,Kolding,Sønderborg,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 21:30:58
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,30. januar 2018,29. juli 2018,597,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",dette er blot en ud af alt for mange anbragte ...,3,Kalundborg,Kalundborg,Kalundborg,Sorø,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 21:31:00
1845,"Statsborgerskab til unge mennesker, som er fød...",30. januar 2018,29. juli 2018,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,"Mennesker som er født og opvokset i Danmark, o...",3,Stevns,København,Næstved,Roskilde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 21:31:02


We have now constructed a dataframe that has the most relevant variables for the borgerforslag dattaset for our analysis. We will now add the scraping time for the data to be able to use for variable generation.

There might have been some problems with the logging function so we check if all rows have scrape times (likely due to a error in fetching the data or similiar):

In [7]:
# Filter the DataFrame to get all rows where the scrape time (Timestamp) is missing
missing_scrape_time_rows = merged_df[merged_df['Scrape'].isna()]

# Display the filtered rows with missing scrape time
missing_scrape_time_rows

,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,Coproposer3_munici,Coproposer4_munici,Coproposer5_munici,Coproposer6_munici,Coproposer7_munici,Coproposer8_munici,Coproposer9_munici,Coproposer10_munici,Scrape
1336,Afskaffelse af bonus- og refusionsordninger,28. januar 2020,26. juli 2020,125,FT-04027,Vi vil foreslå at alle former for bonusordning...,"Kommunerne lader sig styre økonomisk, mht. de ...",4,Varde,Norddjurs,Odense,Esbjerg,Bornholm,NaN,NaN,NaN,NaN,NaN,NaN,NaT


As cells with missing scrape times exist, we impute the missing times by averaging (half the difference) the scrape times from the rows before and after. Since the rows were all scraped with an approximate 1 to 2-second gap, this provides a reasonably accurate estimate. Ultimately, this minor adjustment should not affect the results, as the differences are measured in days, and none of our times cross into a different day.

In [8]:
# First, sort the DataFrame by the relevant columns
merged_df = merged_df.sort_values(by=['File Number', 'Scrape'], ascending=[True, True])

# Step 1: Keep track of rows with missing 'Scrape' times before imputation
missing_scrape_time_rows_before = merged_df[merged_df['Scrape'].isna()].copy()

# Step 2: Impute the missing 'Scrape' times
# Find the index of the rows with missing 'Scrape' time
missing_indices = merged_df[merged_df['Scrape'].isna()].index

# Loop through each missing index and impute the value
for idx in missing_indices:
    if idx > 0 and idx < len(merged_df) - 1:
        time_before = merged_df.loc[idx - 1, 'Scrape']
        time_after = merged_df.loc[idx + 1, 'Scrape']
        
        # Calculate the average time (half the difference)
        avg_time = time_before + (time_after - time_before) / 2
        
        # Fill the missing 'Scrape' time with the calculated average
        merged_df.at[idx, 'Scrape'] = avg_time

# Step 3: Display the rows that were missing 'Scrape' times but now have been imputed
missing_scrape_time_rows_after = merged_df.loc[missing_indices]

# Restore the original index order of the DataFrame
merged_df = merged_df.sort_index()

# Display the DataFrame with the original index order
merged_df

# Display the updated rows
missing_scrape_time_rows_after

,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,Coproposer3_munici,Coproposer4_munici,Coproposer5_munici,Coproposer6_munici,Coproposer7_munici,Coproposer8_munici,Coproposer9_munici,Coproposer10_munici,Scrape
1336,Afskaffelse af bonus- og refusionsordninger,28. januar 2020,26. juli 2020,125,FT-04027,Vi vil foreslå at alle former for bonusordning...,"Kommunerne lader sig styre økonomisk, mht. de ...",4,Varde,Norddjurs,Odense,Esbjerg,Bornholm,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-18 21:15:34


We see no missing scrape times:

In [9]:
# Filter the DataFrame to get all rows where the scrape time (Timestamp) is missing
missing_scrape_time_rows = merged_df[merged_df['Scrape'].isna()]

# Display the filtered rows with missing scrape time
missing_scrape_time_rows

,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,Coproposer3_munici,Coproposer4_munici,Coproposer5_munici,Coproposer6_munici,Coproposer7_munici,Coproposer8_munici,Coproposer9_munici,Coproposer10_munici,Scrape


We can now calculate the age of the borgerforslag:


In [10]:
import pandas as pd

# Create a mapping from Danish month names to English month names
danish_to_english_months = {
    "januar": "January",
    "februar": "February",
    "marts": "March",
    "april": "April",
    "maj": "May",
    "juni": "June",
    "juli": "July",
    "august": "August",
    "september": "September",
    "oktober": "October",
    "november": "November",
    "december": "December"
}

# Function to replace Danish month names with English month names
def replace_danish_months(date_str):
    if isinstance(date_str, str):  # Only apply if the date is a string
        for danish, english in danish_to_english_months.items():
            date_str = date_str.replace(danish, english)
    return date_str

# Assuming merged_df is your DataFrame
# Replace Danish month names with English in 'Start Date' and 'End Date'
merged_df['Start Date'] = merged_df['Start Date'].apply(replace_danish_months)
merged_df['End Date'] = merged_df['End Date'].apply(replace_danish_months)

# Convert the updated 'Start Date' and 'End Date' to datetime
merged_df['Start Date'] = pd.to_datetime(merged_df['Start Date'], format='%d. %B %Y', errors='coerce')
merged_df['End Date'] = pd.to_datetime(merged_df['End Date'], format='%d. %B %Y', errors='coerce')

# Convert 'Scrape' to datetime
merged_df['Scrape'] = pd.to_datetime(merged_df['Scrape'], errors='coerce')

# Calculate the 'Days from start to scrape' in days
merged_df['Days from start to scrape'] = (merged_df['Scrape'] - merged_df['Start Date']).dt.days

# Calculate the 'Days from start to end' in days
merged_df['Days from start to end'] = (merged_df['End Date'] - merged_df['Start Date']).dt.days

# Calculate the 'Lifetime' variable
merged_df['Lifetime'] = merged_df[['Days from start to scrape', 'Days from start to end']].min(axis=1)

# Filter out rows where 'Lifetime' is not 180 or 181 days
# comment this out if graphs etc are wished for non-cleaned data
#merged_df = merged_df[merged_df['Lifetime'].isin([180, 181])]

# Count the number of observations with Lifetime 180 and 181 days
lifetime_counts = merged_df['Lifetime'].value_counts()

# Print the counts
print("Number of observations with Lifetime = 180 days:", lifetime_counts.get(180, 0))
print("Number of observations with Lifetime = 181 days:", lifetime_counts.get(181, 0))

# Constructing monthly dummies
monthly_dummies = pd.get_dummies(merged_df["Start Date"].dt.month, drop_first=True, prefix="Month")

# Adding the monthly dummies to the existing DataFrame
merged_df = pd.concat([merged_df, monthly_dummies], axis=1)

# Display the updated DataFrame
merged_df


Number of observations with Lifetime = 180 days: 1489
Number of observations with Lifetime = 181 days: 201


,Title,Start Date,End Date,Votes,File Number,Main Body,Remarks,Num Coauthors,Proposer1_munici,Coproposer1_munici,...,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,Month_10,Month_11,Month_12
0,Afskaf inklusionsloven,2024-08-15,2025-02-11,39,FT-18153,For at forbedre undervisningen og sikre optima...,"Inklusionsloven, som blev indført i 2012 med d...",3,Aarhus,Aarhus,...,False,False,False,False,False,True,False,False,False,False
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,2025-02-02,82,FT-18099,Forslaget drejer sig om regler for fartbøder v...,Vi synes ikke det er rimeligt at en fartbøde s...,3,Vejle,Vejle,...,False,False,False,False,False,True,False,False,False,False
2,Forbyd dressurridning som konkurrencesport,2024-08-06,2025-02-02,245,FT-18075,Dressurridning som konkurrencesport er en spor...,Dette borgerforslag er stillet at dyreetiske å...,3,Hvidovre,Adressebeskyttelse,...,False,False,False,False,False,True,False,False,False,False
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,2025-02-02,338,FT-18046,Vi stiller et forslag om ændring af reglerne f...,Vi stiller dette forslag for at lette den økon...,3,Randers,København,...,False,False,False,False,False,True,False,False,False,False
4,Bloddonorpligt. (Som værnepligt),2024-07-26,2025-01-22,41,FT-18044,Jeg forslår at der etableres en pligt til at a...,Det er jo velkendt at danske regioner mangler ...,3,Hjørring,Hjørring,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,2018-07-29,189,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,"Af hensyn til forurening.\nAf hensyn til at ""4...",3,Odsherred,Odsherred,...,False,False,False,False,False,False,False,False,False,False
1843,Automatisk førtidspension til personer der har...,2018-01-30,2018-07-29,66,FT-00059,Forslaget er at man automatisk giver førtidspe...,Formålet med lovforslaget er at sikre ofrene f...,4,Vejen,Sønderborg,...,False,False,False,False,False,False,False,False,False,False
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,2018-07-29,597,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",dette er blot en ud af alt for mange anbragte ...,3,Kalundborg,Kalundborg,...,False,False,False,False,False,False,False,False,False,False
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,2018-07-29,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,"Mennesker som er født og opvokset i Danmark, o...",3,Stevns,København,...,False,False,False,False,False,False,False,False,False,False


We now move into cleaning the string data in the main body of text and the remark field. Firstly, we create a a variable that houses both.

In [11]:
import re

# Function to clean text by removing newline characters
def clean_text(text):
    if isinstance(text, str):  # Ensure the input is a string
        text = text.replace('\n', ' ')  # Replace newline characters with a space
        text = text.strip()  # Remove leading and trailing spaces
        text = re.sub(r'<[^>]*>', ' ', text)  # Remove HTML
        text = re.sub(r'[^\w\s,\.]', '', text)  # Remove non-alphanumeric characters, except commas and periods
    return text

# Function to count uppercase letters in a text
def count_uppercase(text):
    return sum(1 for char in text if char.isupper())

# Function to count lowercase letters in a text
def count_lowercase(text):
    return sum(1 for char in text if char.islower())

# Function to calculate LIX score
def calculate_lix(text):
    sentences = re.split(r'[.!?]', text)  # Split text into sentences
    sentences = [s for s in sentences if len(s.strip()) > 0]  # Filter out empty sentences
    num_sentences = len(sentences)
    words = text.split()
    num_words = len(words)
    long_words = [word for word in words if len(word) > 6]
    num_long_words = len(long_words)
    
    if num_sentences == 0:  # To avoid division by zero
        return 0
    
    lix = num_words / num_sentences + (num_long_words / num_words) * 100
    
    # Adjust the LIX score if it falls outside the 25-70 range
    if lix < 25:
        lix = 25
    elif lix > 70:
        lix = 70
    
    return lix

# Combine 'Main Body' and 'Remarks' into the 'Main Body' column itself
merged_df['Main Body'] = merged_df['Main Body'] + ' ' + merged_df['Remarks']

# Drop the 'Remarks' column as it's no longer needed
merged_df = merged_df.drop(columns=['Remarks'])

# Rename the 'Main Body' column to 'Body Text'
merged_df = merged_df.rename(columns={'Main Body': 'Body Text'})

# Apply the cleaning function to 'Title' and 'Body Text'
merged_df['Title'] = merged_df['Title'].apply(clean_text)
merged_df['Body Text'] = merged_df['Body Text'].apply(clean_text)

# Add columns for the length of 'Title' and 'Body Text'
merged_df['Title Length'] = merged_df['Title'].apply(len)
merged_df['Body Text Length'] = merged_df['Body Text'].apply(len)

# Add columns for the number of uppercase and lowercase letters in 'Title' and 'Body Text'
merged_df['Title Uppercase Count'] = merged_df['Title'].apply(count_uppercase)
merged_df['Body Text Uppercase Count'] = merged_df['Body Text'].apply(count_uppercase)

merged_df['Title Lowercase Count'] = merged_df['Title'].apply(count_lowercase)
merged_df['Body Text Lowercase Count'] = merged_df['Body Text'].apply(count_lowercase)

# Calculate the LIX score for the 'Body Text'
merged_df['Body Text LIX Score'] = merged_df['Body Text'].apply(calculate_lix)

# Display the updated DataFrame with the new columns
merged_df


,Title,Start Date,End Date,Votes,File Number,Body Text,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,...,Month_10,Month_11,Month_12,Title Length,Body Text Length,Title Uppercase Count,Body Text Uppercase Count,Title Lowercase Count,Body Text Lowercase Count,Body Text LIX Score
0,Afskaf inklusionsloven,2024-08-15,2025-02-11,39,FT-18153,For at forbedre undervisningen og sikre optima...,3,Aarhus,Aarhus,Esbjerg,...,False,False,False,22,3102,1,18,20,2556,59.349155
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,2025-02-02,82,FT-18099,Forslaget drejer sig om regler for fartbøder v...,3,Vejle,Vejle,Vejle,...,False,False,False,52,1283,2,17,44,999,40.803930
2,Forbyd dressurridning som konkurrencesport,2024-08-06,2025-02-02,245,FT-18075,Dressurridning som konkurrencesport er en spor...,3,Hvidovre,Adressebeskyttelse,Tårnby,...,False,False,False,42,2326,1,40,38,1864,40.113388
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,2025-02-02,338,FT-18046,Vi stiller et forslag om ændring af reglerne f...,3,Randers,København,Randers,...,False,False,False,100,1764,1,14,87,1429,50.907995
4,Bloddonorpligt. Som værnepligt,2024-07-26,2025-01-22,41,FT-18044,Jeg forslår at der etableres en pligt til at a...,3,Hjørring,Hjørring,Hjørring,...,False,False,False,30,702,2,11,25,544,30.552941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,2018-07-29,189,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,3,Odsherred,Odsherred,Odsherred,...,False,False,False,41,545,1,6,33,426,43.400000
1843,Automatisk førtidspension til personer der har...,2018-01-30,2018-07-29,66,FT-00059,Forslaget er at man automatisk giver førtidspe...,4,Vejen,Sønderborg,Sønderborg,...,False,False,False,125,1147,1,6,108,944,53.763418
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,2018-07-29,597,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",3,Kalundborg,Kalundborg,Kalundborg,...,False,False,False,102,1801,88,13,0,1385,33.292443
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,2018-07-29,6.263,FT-00067,Folketinget pålægger regeringen at genindføre ...,3,Stevns,København,Næstved,...,False,False,False,105,4863,2,46,88,3886,49.983954


We also convert the votes variable to english notation:

In [12]:
# Function to convert Danish notation to English notation
def convert_votes_to_english(votes):
    if isinstance(votes, str):  # Ensure the input is a string
        votes = votes.replace('.', '')  # Remove the period used as a thousands separator
        try:
            votes = int(votes)  # Convert the resulting string to an integer
        except ValueError:
            votes = None  # If conversion fails, set the value to None
    return votes

# Apply the conversion function to the 'Votes' column
merged_df['Votes'] = merged_df['Votes'].apply(convert_votes_to_english)

# Display the updated DataFrame with the converted 'Votes' column
merged_df

,Title,Start Date,End Date,Votes,File Number,Body Text,Num Coauthors,Proposer1_munici,Coproposer1_munici,Coproposer2_munici,...,Month_10,Month_11,Month_12,Title Length,Body Text Length,Title Uppercase Count,Body Text Uppercase Count,Title Lowercase Count,Body Text Lowercase Count,Body Text LIX Score
0,Afskaf inklusionsloven,2024-08-15,2025-02-11,39,FT-18153,For at forbedre undervisningen og sikre optima...,3,Aarhus,Aarhus,Esbjerg,...,False,False,False,22,3102,1,18,20,2556,59.349155
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,2025-02-02,82,FT-18099,Forslaget drejer sig om regler for fartbøder v...,3,Vejle,Vejle,Vejle,...,False,False,False,52,1283,2,17,44,999,40.803930
2,Forbyd dressurridning som konkurrencesport,2024-08-06,2025-02-02,245,FT-18075,Dressurridning som konkurrencesport er en spor...,3,Hvidovre,Adressebeskyttelse,Tårnby,...,False,False,False,42,2326,1,40,38,1864,40.113388
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,2025-02-02,338,FT-18046,Vi stiller et forslag om ændring af reglerne f...,3,Randers,København,Randers,...,False,False,False,100,1764,1,14,87,1429,50.907995
4,Bloddonorpligt. Som værnepligt,2024-07-26,2025-01-22,41,FT-18044,Jeg forslår at der etableres en pligt til at a...,3,Hjørring,Hjørring,Hjørring,...,False,False,False,30,702,2,11,25,544,30.552941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,2018-07-29,189,FT-00139,Jeg stiller hermed forslag om at ophævelsen af...,3,Odsherred,Odsherred,Odsherred,...,False,False,False,41,545,1,6,33,426,43.400000
1843,Automatisk førtidspension til personer der har...,2018-01-30,2018-07-29,66,FT-00059,Forslaget er at man automatisk giver førtidspe...,4,Vejen,Sønderborg,Sønderborg,...,False,False,False,125,1147,1,6,108,944,53.763418
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,2018-07-29,597,FT-00034,"Lovforslaget går i sin enkelhed ud på, at sikr...",3,Kalundborg,Kalundborg,Kalundborg,...,False,False,False,102,1801,88,13,0,1385,33.292443
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,2018-07-29,6263,FT-00067,Folketinget pålægger regeringen at genindføre ...,3,Stevns,København,Næstved,...,False,False,False,105,4863,2,46,88,3886,49.983954


Check for what signatures have what lifetimes:

In [13]:
import pandas as pd

# Step 1: Create a new dataset from merged_df to avoid modifying the original
new_df = merged_df.copy()

# Step 2: Convert the 'Votes' column to numeric in the new dataset, forcing any errors to become NaN
new_df['Votes'] = pd.to_numeric(new_df['Votes'], errors='coerce')

# Step 3: Handle NaN values by dropping rows where 'Votes' is NaN in the new dataset
new_df = new_df.dropna(subset=['Votes'])

# Step 4: Create a new column in the new dataset that categorizes 'Votes' based on whether they are below/equal to or above 50,000
new_df['Signatures Category'] = new_df['Votes'].apply(lambda x: 'Below or equal to 50000' if x <= 50000 else 'Above 50000')

# Step 5: Create the cross table using the new dataset
cross_table = pd.crosstab(new_df['Lifetime'], new_df['Signatures Category'])

# Display the cross table
cross_table


Signatures Category,Above 50000,Below or equal to 50000
Lifetime,,
3,0,1
12,0,3
23,0,6
25,0,3
32,0,1
...,...,...
456,1,0
484,1,0
542,1,0


In [14]:
import pandas as pd

# Step 1: Convert the 'Votes' column to numeric, forcing any errors to become NaN
merged_df['Votes'] = pd.to_numeric(merged_df['Votes'], errors='coerce')

# Step 2: Handle NaN values
# Drop rows where 'Votes' is NaN
merged_df = merged_df.dropna(subset=['Votes'])

# Step 3: Create a new column that categorizes 'Votes' based on whether they are below/equal to or above 50,000
merged_df['Signatures Category'] = merged_df['Votes'].apply(lambda x: 'Below or equal to 50000' if x <= 50000 else 'Above 50000')

# Step 4: Create the cross table
cross_table = pd.crosstab(merged_df['Lifetime'], merged_df['Signatures Category'])

# Display the cross table
cross_table


Signatures Category,Above 50000,Below or equal to 50000
Lifetime,,
3,0,1
12,0,3
23,0,6
25,0,3
32,0,1
...,...,...
456,1,0
484,1,0
542,1,0


# Adding Location-features

In [15]:
# Create a list of unique cities

unique_locations = merged_df["Proposer1_munici"].unique()

locations = []

for location in unique_locations:
    if location.strip() != "":
        locations.append(location)

for x in locations:
    print(x)

print(len(locations))

Aarhus
Vejle
Hvidovre
Randers
Hjørring
Slagelse
Svendborg
Odsherred
Guldborgsund
København
Vordingborg
Holstebro
Adressebeskyttelse
Kolding
Horsens
Næstved
Herlev
Kalundborg
Hedensted
Haderslev
Rødovre
Greve
Høje-Taastrup
Ishøj
Lejre
Fredensborg
Odense
Rudersdal
Roskilde
Gribskov
Esbjerg
Favrskov
Kerteminde
Viborg
Hørsholm
Middelfart
Rebild
Silkeborg
Frederiksberg
Nordfyns
Aalborg
Struer
Halsnæs
Hillerød
Langeland
Ballerup
Ringsted
Vejen
Stevns
Tønder
Sønderborg
Køge
Gladsaxe
Faxe
Holbæk
Bornholm
Brøndby
Solrød
Helsingør
Norddjurs
Vallensbæk
Mariagerfjord
Frederikssund
Gentofte
Herning
Furesø
Fredericia
Odder
Sorø
Ringkøbing-Skjern
Frederikshavn
Skanderborg
Skive
Ærø
Aabenraa
Lyngby-Taarbæk
Tårnby
Faaborg-Midtfyn
Vesthimmerlands
Nyborg
Albertslund
Thisted
Not provided
Dragør
Allerød
Morsø
Egedal
Billund
Varde
Syddjurs
Glostrup
Assens
Jammerbugt
Lolland
Ikast-Brande
Læsø
Samsø
Lemvig
Brønderslev
Sermersooq
Holbæk Kommune
101


In [16]:
# Get the coordinates of the cities from an API

import requests
response = requests.get("https://api.dataforsyningen.dk/kommuner")

cities_with_coordinates = pd.DataFrame(columns=["Proposer1_munici", "Latitude", "Longitude"])

if response.status_code == 200:
    data = response.json()

    for item in data:
        new_row = {"Proposer1_munici": item["navn"], 
                   "Latitude": item["visueltcenter"][0], 
                   "Longitude": item["visueltcenter"][1]}
        
        new_row_df = pd.DataFrame([new_row])
        cities_with_coordinates = pd.concat([cities_with_coordinates, new_row_df], ignore_index=True)
else:
    print(response.status_code)

print(cities_with_coordinates)

   Proposer1_munici   Latitude  Longitude
0         København  12.493909  55.704091
1     Frederiksberg  12.523733  55.679365
2          Ballerup  12.368516  55.727075
3           Brøndby  12.404382  55.645037
4            Dragør  12.650228  55.593807
..              ...        ...        ...
94           Rebild   9.760377  56.869039
95    Mariagerfjord   9.925576  56.733570
96       Jammerbugt   9.621991  57.165636
97          Aalborg   9.997100  56.980870
98         Hjørring  10.077907  57.468403

[99 rows x 3 columns]


/var/folders/s3/s58q439926xb4bq1gzzpvw0r0000gn/T/ipykernel_73809/334150488.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cities_with_coordinates = pd.concat([cities_with_coordinates, new_row_df], ignore_index=True)


In [17]:
# Only keep cities in dataframe that exist in the list

# Filter the DataFrame
cities_filtered = cities_with_coordinates[cities_with_coordinates['Proposer1_munici'].isin(locations)]
cities_filtered

,Proposer1_munici,Latitude,Longitude
0,København,12.493909,55.704091
1,Frederiksberg,12.523733,55.679365
2,Ballerup,12.368516,55.727075
3,Brøndby,12.404382,55.645037
4,Dragør,12.650228,55.593807
...,...,...,...
94,Rebild,9.760377,56.869039
95,Mariagerfjord,9.925576,56.733570
96,Jammerbugt,9.621991,57.165636
97,Aalborg,9.997100,56.980870


In [18]:
# Coordinates
coordinates_danish_parliament = (12.580217, 55.676308)

# Import geodesic
# !pip install geopy
from geopy.distance import geodesic


def calculate_distance_to_parliament(latitude, longitude):

    coordinates_city = (latitude, longitude)

    distance = geodesic(coordinates_city, coordinates_danish_parliament).kilometers

    return distance



# Apply the function to create a new column 'z'
cities_filtered['Distance to parliament km'] = cities_filtered.apply(lambda row: calculate_distance_to_parliament(row['Latitude'], row['Longitude']), axis=1)


cities_filtered

/var/folders/s3/s58q439926xb4bq1gzzpvw0r0000gn/T/ipykernel_73809/1617326927.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_filtered['Distance to parliament km'] = cities_filtered.apply(lambda row: calculate_distance_to_parliament(row['Latitude'], row['Longitude']), axis=1)


,Proposer1_munici,Latitude,Longitude,Distance to parliament km
0,København,12.493909,55.704091,10.014071
1,Frederiksberg,12.523733,55.679365,6.257462
2,Ballerup,12.368516,55.727075,24.061125
3,Brøndby,12.404382,55.645037,19.746730
4,Dragør,12.650228,55.593807,11.846270
...,...,...,...,...
94,Rebild,9.760377,56.869039,338.026358
95,Mariagerfjord,9.925576,56.733570,315.522035
96,Jammerbugt,9.621991,57.165636,365.437669
97,Aalborg,9.997100,56.980870,319.262282


In [19]:
import pandas as pd

# Merge merged_df onto cities_filtered with merged_df being the primary DataFrame
merged_df = pd.merge(merged_df, cities_filtered, on='Proposer1_munici', how='left')


# This next part is added to lower the number of missing value for "Distance to parliament km"
# This is done by calculating the distance from the co-authors to parliament if the primary author has no adress

# Import math
import math

def add_co_proposer_location(co_proposer_location_key):

    # To add distance to missing values using the first co-author on the list
    for index, row in merged_df.iterrows():
        if math.isnan(row["Distance to parliament km"]):

            municipality = cities_filtered[cities_filtered['Proposer1_munici'] == row[co_proposer_location_key]]

            # Continue if there are no municipalities that match
            if municipality.shape[0] != 0:
                # Use the first municipality
                municipality_first = municipality.iloc[0]
                merged_df.at[index, "Distance to parliament km"] = municipality_first["Distance to parliament km"]

# Adds co_proposer distance to dataframe if no existing distance is found
add_co_proposer_location("Coproposer1_munici")
add_co_proposer_location("Coproposer2_munici")
add_co_proposer_location("Coproposer3_munici")
add_co_proposer_location("Coproposer4_munici")
add_co_proposer_location("Coproposer5_munici")
add_co_proposer_location("Coproposer6_munici")
add_co_proposer_location("Coproposer7_munici")
add_co_proposer_location("Coproposer8_munici")
add_co_proposer_location("Coproposer9_munici")

# Show dataframe
merged_df

# Display the resulting DataFrame
print(merged_df)

                                                  Title Start Date   End Date  \
0                                Afskaf inklusionsloven 2024-08-15 2025-02-11   
1     Fartbøder forhindrer i at opnå Dansk statsborg... 2024-08-06 2025-02-02   
2            Forbyd dressurridning som konkurrencesport 2024-08-06 2025-02-02   
3     Lavere skat og fjernelse af minimumsalder på p... 2024-08-06 2025-02-02   
4                        Bloddonorpligt. Som værnepligt 2024-07-26 2025-01-22   
...                                                 ...        ...        ...   
1842          Opæhvelse af begrænsning på 45 knallerter 2018-01-30 2018-07-29   
1843  Automatisk førtidspension til personer der har... 2018-01-30 2018-07-29   
1844  POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ... 2018-01-30 2018-07-29   
1845  Statsborgerskab til unge mennesker, som er fød... 2018-01-30 2018-07-29   
1846           Forslag om at afskaffe uddannelsesloftet 2018-01-26 2018-05-15   

      Votes File Number    

In [20]:
# Now we make a boolean value to indicate if a CI is from one of the four big cities in Denmark (Aalborg, Aarhus, Odense, København)

list_of_municipalities = ["Aalborg", "Aarhus", "Odense", "København"]

# Create a boolean indicating if a CI is from one of the four big cities
merged_df['Big city'] = merged_df['Proposer1_munici'].isin(list_of_municipalities)

We now drop the columns we dont need for further analysis:

In [21]:
# Drop specific columns
columns_to_drop = ['Longitude', 
                   'Latitude', 
                   #'Start Date', 
                   #'Lifetime',
                   'End Date', 
                   'Days from start to scrape',
                   'Days from start to end',
                   'File Number', 
                   'Proposer1_munici', 
                   'Scrape']

# Add columns that match the pattern 'CoproposerX_munici'
columns_to_drop += merged_df.filter(regex=r'^Coproposer\d+_munici$').columns.tolist()

# Drop the columns from merged_df
merged_df = merged_df.drop(columns=columns_to_drop)

# Display the resulting DataFrame
merged_df


,Title,Start Date,Votes,Body Text,Num Coauthors,Lifetime,Month_2,Month_3,Month_4,Month_5,...,Title Length,Body Text Length,Title Uppercase Count,Body Text Uppercase Count,Title Lowercase Count,Body Text Lowercase Count,Body Text LIX Score,Signatures Category,Distance to parliament km,Big city
0,Afskaf inklusionsloven,2024-08-15,39,For at forbedre undervisningen og sikre optima...,3,3,False,False,False,False,...,22,3102,1,18,20,2556,59.349155,Below or equal to 50000,278.896623,True
1,Fartbøder forhindrer i at opnå Dansk statsborg...,2024-08-06,82,Forslaget drejer sig om regler for fartbøder v...,3,12,False,False,False,False,...,52,1283,2,17,44,999,40.803930,Below or equal to 50000,355.768312,False
2,Forbyd dressurridning som konkurrencesport,2024-08-06,245,Dressurridning som konkurrencesport er en spor...,3,12,False,False,False,False,...,42,2326,1,40,38,1864,40.113388,Below or equal to 50000,13.285377,False
3,Lavere skat og fjernelse af minimumsalder på p...,2024-08-06,338,Vi stiller et forslag om ændring af reglerne f...,3,12,False,False,False,False,...,100,1764,1,14,87,1429,50.907995,Below or equal to 50000,291.079559,False
4,Bloddonorpligt. Som værnepligt,2024-07-26,41,Jeg forslår at der etableres en pligt til at a...,3,23,False,False,False,False,...,30,702,2,11,25,544,30.552941,Below or equal to 50000,338.944018,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,Opæhvelse af begrænsning på 45 knallerter,2018-01-30,189,Jeg stiller hermed forslag om at ophævelsen af...,3,180,False,False,False,False,...,41,545,1,6,33,426,43.400000,Below or equal to 50000,110.013938,False
1843,Automatisk førtidspension til personer der har...,2018-01-30,66,Forslaget er at man automatisk giver førtidspe...,4,180,False,False,False,False,...,125,1147,1,6,108,944,53.763418,Below or equal to 50000,390.978286,False
1844,POLITISK FORSLAG TIL HØRING I FOLKETINGET OM Æ...,2018-01-30,597,"Lovforslaget går i sin enkelhed ud på, at sikr...",3,180,False,False,False,False,...,102,1801,88,13,0,1385,33.292443,Below or equal to 50000,149.484204,False
1845,"Statsborgerskab til unge mennesker, som er fød...",2018-01-30,6263,Folketinget pålægger regeringen at genindføre ...,3,180,False,False,False,False,...,105,4863,2,46,88,3886,49.983954,Below or equal to 50000,46.247679,False


## Additional Title Variables

In [22]:
# Create a dictionary comprehensions in which we iterate over each title/document
title_var_caps = [document.isupper() for document in merged_df["Title"]] # Title in full uppercase-variable
merged_df['Title All Uppercase'] = title_var_caps

In [23]:
# Initialize Afinn object for sentiment analysis
afn = Afinn(language='da')

# creating Sentiment variable on Body Text
body_var_sentiment_categorical = [
    "negative" if score < 0 else ("positive" if score > 0 else "neutral")
    for score in [afn.score(document) for document in merged_df["Body Text"]]
]

# Constructing sentiment dummies
sentiment_dummies = pd.get_dummies(body_var_sentiment_categorical, drop_first=True, prefix="Sentiment")

# Adding the sentiment dummies to the existing DataFrame
merged_df = pd.concat([merged_df, sentiment_dummies], axis=1)

Output dataset:

In [24]:
# Specify the path where you want to save the CSV file
output_path = 'Data/df_after_variable.csv'

# Save the DataFrame to the CSV file
merged_df.to_csv(output_path, index=False)

# Print the output path to confirm where the file was saved
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: Data/df_after_variable.csv
